# Peak Shaving mit Batteriespeichern – Lösung und Demonstration

Dieses Notebook zeigt den Use-Case **Peak Shaving** bzw. **Lastspitzenkappung** anhand eines realen Jahresprofils der Hochschule Kempten. Es ist nicht als Aufgabenblatt aufgebaut, sondern als durchgängige, ausführbare Demonstration.

Im Mittelpunkt steht die Frage, wie ein Batteriespeicher so geladen und entladen werden kann, dass die aus dem Netz bezogene **Residuallast** reduziert wird. Dadurch sinkt die abrechnungsrelevante Jahreshöchstleistung. Gleichzeitig entstehen Investitionskosten für den Speicher.

Das Notebook besteht aus drei Kapiteln:

1. **Feste Peak-Shaving-Grenze:** Der Nutzer gibt eine maximale Netzbezugsleistung vor. Die Optimierung bestimmt die minimale passende Speicherleistung, Speicherkapazität und Betriebsstrategie.
2. **Optimale Peak-Shaving-Grenze:** Die Peak-Shaving-Grenze wird selbst zur Optimierungsvariable. Der Optimierer wählt Speichergröße, Speicherbetrieb und optimale Restspitze auf Basis der Kostenstruktur.
3. **Technologievergleich:** Zwei Speichertechnologien werden mit unterschiedlichen Wirkungsgraden und Kostenparametern verglichen, angelehnt an Lithium-Ionen- und Redox-Flow-Batterien.

Alle Modelle sind linear formuliert und werden mit **Pyomo** und dem **HiGHS-Solver** gelöst. Die Zellen sind so aufgebaut, dass sie in **Deepnote** funktionieren, wenn die Datei `HSK_22_1h.csv` im Projektverzeichnis liegt.


## Vorbereitung: Pakete, Solver und Daten

Die folgende Installationszelle ist für Deepnote vorgesehen. Sie installiert Pyomo, HiGHS sowie die benötigten Standardpakete. Wenn die Pakete bereits vorhanden sind, wird die Installation einfach übersprungen bzw. aktualisiert.


In [ ]:
%pip install -q pyomo highspy pandas plotly


In [ ]:
from pathlib import Path
import math

import pandas as pd
import pyomo.environ as opt
from pyomo.opt import TerminationCondition
import plotly.graph_objects as go

pd.options.plotting.backend = "plotly"
template = "plotly_white"


### Hilfsfunktion für den HiGHS-Solver

Pyomo kann HiGHS je nach Umgebung unter unterschiedlichen Solver-Namen ansprechen. In Deepnote funktioniert in der Regel `appsi_highs`, sobald `highspy` installiert ist. Die folgende Funktion sucht automatisch nach einer verfügbaren HiGHS-Schnittstelle.


In [ ]:
def highs_solver():
    """Erzeugt einen verfügbaren HiGHS-Solver für Pyomo."""
    for solver_name in ["appsi_highs", "highs"]:
        solver = opt.SolverFactory(solver_name)
        if solver.available(exception_flag=False):
            print(f"Verwende Solver: {solver_name}")
            return solver
    raise RuntimeError(
        "HiGHS ist nicht verfügbar. Bitte ausführen: %pip install pyomo highspy"
    )

solver = highs_solver()


### Daten einlesen

Das Profil enthält drei zentrale Leistungszeitreihen:

- `pUser`: elektrische Last des Standorts,
- `pPV`: PV-Erzeugung,
- `pRes`: Residuallast, also die nach PV verbleibende Leistung, die aus dem Netz gedeckt werden muss.

Für das Peak Shaving wird im Folgenden `pRes` optimiert. Das Profil liegt in stündlicher Auflösung vor. Da die Datei einen zusätzlichen Zeitpunkt `2023-01-01 00:00:00` enthält, wird für das Beispieljahr nur der Zeitraum bis `2022-12-31 23:00:00` genutzt.


In [ ]:
def lade_hsk_profil():
    """Lädt das HSK-Jahresprofil aus typischen lokalen und Deepnote-Pfaden."""
    kandidaten = [
        Path("HSK_22_1h.csv"),
        Path("../data/HSK_22_1h.csv"),
        Path("..") / "data" / "HSK" / "HSK_22_1h.csv",
    ]
    for pfad in kandidaten:
        if pfad.exists():
            print(f"Lade Profil aus: {pfad}")
            df = pd.read_csv(pfad, index_col="time_date", parse_dates=True)
            break
    else:
        raise FileNotFoundError(
            "Die Datei HSK_22_1h.csv wurde nicht gefunden. "
            "Bitte die CSV-Datei in das gleiche Verzeichnis wie dieses Notebook legen."
        )

    # Für ein vollständiges Beispieljahr werden genau die 8760 Stunden von 2022 genutzt.
    df = df.loc["2022-01-01 00:00:00":"2022-12-31 23:00:00"].copy()
    df = df[["pRes", "pPV", "pUser"]].astype(float)
    return df

profile = lade_hsk_profil()
dt = 1.0  # h, stündliche Auflösung des Profils
profile.head()


### Ausgangssituation: PV, Last und Residuallast

Die erste Darstellung bleibt bewusst nah am Ausgangsnotebook: PV-Erzeugung, Standortlast und Residuallast werden gemeinsam dargestellt. Die Residuallast ist die relevante Größe für die Netzbezugsleistung und damit für das Peak Shaving.


In [ ]:
profil_plot = profile.rename(
    columns={
        "pPV": "PV-Erzeugung [kW]",
        "pUser": "Last [kW]",
        "pRes": "Residuallast [kW]",
    }
)
profil_plot[["PV-Erzeugung [kW]", "Last [kW]", "Residuallast [kW]"]].plot(
    template=template,
    labels={"value": "Leistung [kW]", "time_date": "Zeit", "variable": "Zeitreihe"},
    title="PV, Last und Residuallast des Beispieljahres",
)


### Kosten- und Technologieparameter

Die Stromkosten bestehen aus einem Arbeitspreis und einem Leistungspreis. Der Arbeitspreis wird auf jede bezogene Kilowattstunde angewendet. Der Leistungspreis wird auf die maximale Netzbezugsleistung des Jahres angewendet.

Die Speicherkosten können vom Nutzer frei vorgegeben werden und bestehen aus:

- **Fixkosten** in Euro,
- **spezifischen Kapazitätskosten** in Euro pro kWh installierter Speicherkapazität,
- **spezifischen Leistungskosten** in Euro pro kW installierter Speicherleistung.

Für die Optimierung in Kapitel 2 und 3 werden die Investitionskosten über eine Annuität in jährliche Kosten umgerechnet. Für die einfache Amortisationsrechnung wird zusätzlich betrachtet, nach wie vielen Jahren die reine Stromkostenersparnis die Investitionskosten kompensiert.


In [ ]:
# Stromkosten des Beispielstandorts
stromkosten = {
    "arbeitspreis_eur_kwh": 0.12,       # €/kWh
    "leistungspreis_eur_kw_a": 200.0,  # €/kW und Jahr
}

# Vom Nutzer anpassbare Speicherkosten und technische Parameter
speicher_standard = {
    "fixkosten_eur": 50_000.0,                 # €
    "kapazitaetskosten_eur_kwh": 300.0,        # €/kWh
    "leistungskosten_eur_kw": 150.0,           # €/kW
    "soc_bounds": (0.10, 0.90),                # zulässiger SOC-Bereich
    "soc_start": 0.10,                         # Start-/End-SOC relativ zu E_N
    "eta_laden": 0.95,
    "eta_entladen": 0.95,
    "nutzungsdauer_jahre": 20,
    "kalkulationszins": 0.05,
}

# Obere Grenzen für die Dimensionierungsvariablen. Diese Werte dürfen bewusst großzügig sein.
dimensionierungsgrenzen = {
    "max_leistung_kw": 1_000.0,
    "max_kapazitaet_kwh": 50_000.0,
}

# Nutzerseitig vorgegebene Peak-Shaving-Grenze für Kapitel 1
peak_shaving_grenze_kw = 350.0


### Kostenfunktionen

Die folgenden Funktionen werden in allen Kapiteln verwendet. Sie berechnen die Stromrechnung ohne und mit Speicher, die Investitionskosten des Speichers, die annualisierten Speicherkosten sowie eine einfache statische Amortisationsdauer.


In [ ]:
def annuitaetsfaktor(zinssatz, nutzungsdauer_jahre):
    """Annuitätsfaktor zur Umrechnung einer Investition in jährliche Kosten."""
    if zinssatz == 0:
        return 1 / nutzungsdauer_jahre
    q = 1 + zinssatz
    return (q**nutzungsdauer_jahre * zinssatz) / (q**nutzungsdauer_jahre - 1)


def stromrechnung(netzbezug_kw, kosten, dt_h):
    """Berechnet Arbeitskosten, Leistungskosten und Gesamtkosten für ein Leistungsprofil."""
    energie_kwh = float(netzbezug_kw.sum() * dt_h)
    peak_kw = float(netzbezug_kw.max())
    arbeitskosten = energie_kwh * kosten["arbeitspreis_eur_kwh"]
    leistungskosten = peak_kw * kosten["leistungspreis_eur_kw_a"]
    return {
        "energie_kwh": energie_kwh,
        "peak_kw": peak_kw,
        "arbeitskosten_eur_a": arbeitskosten,
        "leistungskosten_eur_a": leistungskosten,
        "stromkosten_eur_a": arbeitskosten + leistungskosten,
    }


def investitionskosten_speicher(kapazitaet_kwh, leistung_kw, params, tol=1e-6):
    """Berechnet CAPEX des Speichers aus Fix-, Kapazitäts- und Leistungskosten."""
    installiert = (kapazitaet_kwh > tol) or (leistung_kw > tol)
    fix = params["fixkosten_eur"] if installiert else 0.0
    return (
        fix
        + params["kapazitaetskosten_eur_kwh"] * kapazitaet_kwh
        + params["leistungskosten_eur_kw"] * leistung_kw
    )


def jahreskosten_speicher(kapazitaet_kwh, leistung_kw, params):
    """Berechnet annualisierte Speicherkosten."""
    capex = investitionskosten_speicher(kapazitaet_kwh, leistung_kw, params)
    af = annuitaetsfaktor(params["kalkulationszins"], params["nutzungsdauer_jahre"])
    return capex * af


def amortisationsdauer_jahre(investition_eur, jaehrliche_stromkostenersparnis_eur_a):
    """Statische Amortisationsdauer in Jahren."""
    if jaehrliche_stromkostenersparnis_eur_a <= 0:
        return math.inf
    return investition_eur / jaehrliche_stromkostenersparnis_eur_a


def drucke_wirtschaftlichkeit(name, ohne_speicher, mit_speicher, kapazitaet_kwh, leistung_kw, params):
    """Erzeugt eine kompakte Ergebnisübersicht."""
    capex = investitionskosten_speicher(kapazitaet_kwh, leistung_kw, params)
    annualisierte_speicherkosten = jahreskosten_speicher(kapazitaet_kwh, leistung_kw, params)
    stromkostenersparnis = ohne_speicher["stromkosten_eur_a"] - mit_speicher["stromkosten_eur_a"]
    netto_vorteil = stromkostenersparnis - annualisierte_speicherkosten
    amortisation = amortisationsdauer_jahre(capex, stromkostenersparnis)

    print(f"--- {name} ---")
    print(f"Installierte Speicherkapazität: {kapazitaet_kwh:,.2f} kWh")
    print(f"Installierte Speicherleistung:  {leistung_kw:,.2f} kW")
    print(f"Investitionskosten Speicher:    {capex:,.2f} €")
    print(f"Annualisierte Speicherkosten:   {annualisierte_speicherkosten:,.2f} €/a")
    print()
    print(f"Peak ohne Speicher:             {ohne_speicher['peak_kw']:,.2f} kW")
    print(f"Peak mit Speicher:              {mit_speicher['peak_kw']:,.2f} kW")
    print(f"Stromkosten ohne Speicher:      {ohne_speicher['stromkosten_eur_a']:,.2f} €/a")
    print(f"Stromkosten mit Speicher:       {mit_speicher['stromkosten_eur_a']:,.2f} €/a")
    print(f"Jährliche Stromkostenersparnis: {stromkostenersparnis:,.2f} €/a")
    print(f"Netto-Vorteil nach Annuität:    {netto_vorteil:,.2f} €/a")
    if math.isfinite(amortisation):
        print(f"Statische Amortisationsdauer:   {amortisation:,.1f} Jahre")
    else:
        print("Statische Amortisationsdauer:   keine Amortisation bei diesen Parametern")

    return {
        "capex_eur": capex,
        "annualisierte_speicherkosten_eur_a": annualisierte_speicherkosten,
        "stromkostenersparnis_eur_a": stromkostenersparnis,
        "netto_vorteil_eur_a": netto_vorteil,
        "amortisationsdauer_jahre": amortisation,
    }

kosten_ohne_speicher = stromrechnung(profile["pRes"], stromkosten, dt)
kosten_ohne_speicher


## Kapitel 1 – Feste Peak-Shaving-Grenze

In diesem Kapitel wird die maximale Netzbezugsleistung durch den Nutzer vorgegeben. Die Optimierung bestimmt dann die **kleinste bzw. kostengünstigste Speicherkonfiguration**, mit der diese Grenze eingehalten werden kann.

Die Speichergröße ist hier also nicht vorgegeben, sondern Ergebnis der Optimierung:

- installierte Speicherleistung $P_N$,
- installierte Speicherkapazität $E_N$,
- Ladeleistung $p^{ch}_t$,
- Entladeleistung $p^{dis}_t$,
- Energieinhalt $E_t$,
- Netzbezug $p^{grid}_t$.

Die vorgegebene Grenze lautet:

$$
p^{grid}_t \leq p^{limit} \quad \forall t
$$

Als Zielgröße wird die Investition des Speichers minimiert. Die Fixkosten werden über eine binäre Installationsvariable berücksichtigt. Dadurch bleibt das Modell linear, wird aber ein gemischt-ganzzahliges lineares Programm. HiGHS kann solche Modelle lösen.


In [ ]:
def build_model_feste_peakgrenze(load_profile, peak_limit_kw, speicher_params, grenzen, dt_h):
    """Dimensioniert den minimalen Speicher zur Einhaltung einer festen Peak-Shaving-Grenze."""
    last = list(load_profile["pRes"].astype(float).values)
    n = len(last)
    soc_min, soc_max = speicher_params["soc_bounds"]
    soc_start = speicher_params["soc_start"]
    eta_c = speicher_params["eta_laden"]
    eta_d = speicher_params["eta_entladen"]

    model = opt.ConcreteModel()
    model.T = opt.RangeSet(0, n - 1)
    model.p_load = opt.Param(model.T, initialize={t: last[t] for t in range(n)})

    # Betriebsvariablen
    model.p_ch = opt.Var(model.T, within=opt.NonNegativeReals)
    model.p_dis = opt.Var(model.T, within=opt.NonNegativeReals)
    model.e_bess = opt.Var(model.T, within=opt.NonNegativeReals)
    model.p_grid = opt.Var(model.T, within=opt.NonNegativeReals)

    # Dimensionierungsvariablen
    model.E_N = opt.Var(within=opt.NonNegativeReals, bounds=(0, grenzen["max_kapazitaet_kwh"]))
    model.P_N = opt.Var(within=opt.NonNegativeReals, bounds=(0, grenzen["max_leistung_kw"]))
    model.storage_installed = opt.Var(within=opt.Binary)

    # Kopplung der Installationsentscheidung mit Dimensionierungsgrenzen
    model.capacity_installation_limit = opt.Constraint(
        expr=model.E_N <= grenzen["max_kapazitaet_kwh"] * model.storage_installed
    )
    model.power_installation_limit = opt.Constraint(
        expr=model.P_N <= grenzen["max_leistung_kw"] * model.storage_installed
    )

    # Speicherleistungsgrenzen
    model.charge_power_limit = opt.Constraint(model.T, rule=lambda m, t: m.p_ch[t] <= m.P_N)
    model.discharge_power_limit = opt.Constraint(model.T, rule=lambda m, t: m.p_dis[t] <= m.P_N)

    # Energieinhalt und SOC-Grenzen
    model.energy_upper = opt.Constraint(model.T, rule=lambda m, t: m.e_bess[t] <= soc_max * m.E_N)
    model.energy_lower = opt.Constraint(model.T, rule=lambda m, t: m.e_bess[t] >= soc_min * m.E_N)

    def energy_balance(m, t):
        delta_e = (m.p_ch[t] * eta_c - m.p_dis[t] / eta_d) * dt_h
        if t == 0:
            return m.e_bess[t] == soc_start * m.E_N + delta_e
        return m.e_bess[t] == m.e_bess[t - 1] + delta_e

    model.energy_balance = opt.Constraint(model.T, rule=energy_balance)
    model.end_soc = opt.Constraint(expr=model.e_bess[n - 1] >= soc_start * model.E_N)

    # Netzbilanz und feste Peak-Shaving-Grenze
    model.grid_balance = opt.Constraint(
        model.T, rule=lambda m, t: m.p_grid[t] == m.p_load[t] + m.p_ch[t] - m.p_dis[t]
    )
    model.fixed_peak_limit = opt.Constraint(model.T, rule=lambda m, t: m.p_grid[t] <= peak_limit_kw)

    # Ziel: kostengünstigste Speichergröße zur Einhaltung der vorgegebenen Grenze
    model.obj = opt.Objective(
        expr=(
            speicher_params["fixkosten_eur"] * model.storage_installed
            + speicher_params["kapazitaetskosten_eur_kwh"] * model.E_N
            + speicher_params["leistungskosten_eur_kw"] * model.P_N
        ),
        sense=opt.minimize,
    )
    return model


In [ ]:
def loese_modell(model, tee=False):
    """Löst ein Pyomo-Modell mit HiGHS und prüft den Solverstatus."""
    ergebnis = solver.solve(model, tee=tee)
    termination = ergebnis.solver.termination_condition
    if termination != TerminationCondition.optimal:
        raise RuntimeError(f"Optimierung nicht optimal beendet. Status: {termination}")
    return ergebnis


def ergebnisse_aus_modell(model, load_profile):
    """Überführt Pyomo-Ergebnisse in einen DataFrame."""
    df = load_profile.copy()
    df["Laden_kW"] = [opt.value(model.p_ch[t]) for t in model.T]
    df["Entladen_kW"] = [opt.value(model.p_dis[t]) for t in model.T]
    df["BESS_Leistung_kW"] = df["Laden_kW"] - df["Entladen_kW"]
    df["BESS_Energie_kWh"] = [opt.value(model.e_bess[t]) for t in model.T]
    df["Netzbezug_kW"] = [opt.value(model.p_grid[t]) for t in model.T]
    return df


def plot_peak_shaving_ergebnis(df, titel):
    """Erzeugt die drei Standardplots für Peak Shaving."""
    fig1 = df[["pRes", "Netzbezug_kW"]].rename(
        columns={"pRes": "Residuallast ohne Speicher [kW]", "Netzbezug_kW": "Netzbezug mit Speicher [kW]"}
    ).plot(
        template=template,
        labels={"value": "Leistung [kW]", "time_date": "Zeit", "variable": "Zeitreihe"},
        title=f"{titel}: Netzbezug vor und nach Peak Shaving",
    )
    fig1.show()

    fig2 = df[["BESS_Leistung_kW"]].plot(
        template=template,
        labels={"value": "Leistung [kW]", "time_date": "Zeit", "variable": "Zeitreihe"},
        title=f"{titel}: Speicherleistung, positiv = Laden, negativ = Entladen",
    )
    fig2.show()

    fig3 = df[["BESS_Energie_kWh"]].plot(
        template=template,
        labels={"value": "Energie [kWh]", "time_date": "Zeit", "variable": "Zeitreihe"},
        title=f"{titel}: Energieinhalt des Speichers",
    )
    fig3.show()


In [ ]:
model_kapitel_1 = build_model_feste_peakgrenze(
    load_profile=profile,
    peak_limit_kw=peak_shaving_grenze_kw,
    speicher_params=speicher_standard,
    grenzen=dimensionierungsgrenzen,
    dt_h=dt,
)
loese_modell(model_kapitel_1)

res_kapitel_1 = ergebnisse_aus_modell(model_kapitel_1, profile)

kapitel_1_kapazitaet = opt.value(model_kapitel_1.E_N)
kapitel_1_leistung = opt.value(model_kapitel_1.P_N)
kapitel_1_kosten_mit_speicher = stromrechnung(res_kapitel_1["Netzbezug_kW"], stromkosten, dt)

wirtschaftlichkeit_kapitel_1 = drucke_wirtschaftlichkeit(
    name=f"Kapitel 1: feste Peak-Shaving-Grenze von {peak_shaving_grenze_kw:.1f} kW",
    ohne_speicher=kosten_ohne_speicher,
    mit_speicher=kapitel_1_kosten_mit_speicher,
    kapazitaet_kwh=kapitel_1_kapazitaet,
    leistung_kw=kapitel_1_leistung,
    params=speicher_standard,
)


In [ ]:
plot_peak_shaving_ergebnis(res_kapitel_1, "Kapitel 1")


## Kapitel 2 – Optimale Peak-Shaving-Grenze als Ergebnis der Optimierung

Im ersten Kapitel war die Peak-Shaving-Grenze durch den Nutzer vorgegeben. In diesem Kapitel wird dieser Freiheitsgrad geöffnet: Der Optimierer entscheidet selbst, wie stark die Lastspitze reduziert werden soll.

Dazu wird die maximale Netzbezugsleistung $$p^{peak}$$ als Variable modelliert:

$$
p^{grid}_t \leq p^{peak} \quad \forall t
$$

Die Zielfunktion minimiert die Summe aus jährlichen Stromkosten und annualisierten Speicherkosten:

$$
\min \; C^{energy} + C^{peak} + C^{storage,annual}
$$

Dadurch kann die Optimierung abwägen: Eine stärkere Lastspitzenkappung senkt den Leistungspreis, erfordert aber größere und damit teurere Speicher.


In [ ]:
def build_model_optimale_peakgrenze(load_profile, stromkosten, speicher_params, grenzen, dt_h):
    """Optimiert Speichergröße, Betrieb und Peak-Shaving-Grenze gemeinsam."""
    last = list(load_profile["pRes"].astype(float).values)
    n = len(last)
    soc_min, soc_max = speicher_params["soc_bounds"]
    soc_start = speicher_params["soc_start"]
    eta_c = speicher_params["eta_laden"]
    eta_d = speicher_params["eta_entladen"]
    af = annuitaetsfaktor(speicher_params["kalkulationszins"], speicher_params["nutzungsdauer_jahre"])

    model = opt.ConcreteModel()
    model.T = opt.RangeSet(0, n - 1)
    model.p_load = opt.Param(model.T, initialize={t: last[t] for t in range(n)})

    # Betriebsvariablen
    model.p_ch = opt.Var(model.T, within=opt.NonNegativeReals)
    model.p_dis = opt.Var(model.T, within=opt.NonNegativeReals)
    model.e_bess = opt.Var(model.T, within=opt.NonNegativeReals)
    model.p_grid = opt.Var(model.T, within=opt.NonNegativeReals)

    # Dimensionierung und optimale Peak-Grenze
    model.E_N = opt.Var(within=opt.NonNegativeReals, bounds=(0, grenzen["max_kapazitaet_kwh"]))
    model.P_N = opt.Var(within=opt.NonNegativeReals, bounds=(0, grenzen["max_leistung_kw"]))
    model.p_peak = opt.Var(within=opt.NonNegativeReals)
    model.storage_installed = opt.Var(within=opt.Binary)

    model.capacity_installation_limit = opt.Constraint(
        expr=model.E_N <= grenzen["max_kapazitaet_kwh"] * model.storage_installed
    )
    model.power_installation_limit = opt.Constraint(
        expr=model.P_N <= grenzen["max_leistung_kw"] * model.storage_installed
    )

    model.charge_power_limit = opt.Constraint(model.T, rule=lambda m, t: m.p_ch[t] <= m.P_N)
    model.discharge_power_limit = opt.Constraint(model.T, rule=lambda m, t: m.p_dis[t] <= m.P_N)
    model.energy_upper = opt.Constraint(model.T, rule=lambda m, t: m.e_bess[t] <= soc_max * m.E_N)
    model.energy_lower = opt.Constraint(model.T, rule=lambda m, t: m.e_bess[t] >= soc_min * m.E_N)

    def energy_balance(m, t):
        delta_e = (m.p_ch[t] * eta_c - m.p_dis[t] / eta_d) * dt_h
        if t == 0:
            return m.e_bess[t] == soc_start * m.E_N + delta_e
        return m.e_bess[t] == m.e_bess[t - 1] + delta_e

    model.energy_balance = opt.Constraint(model.T, rule=energy_balance)
    model.end_soc = opt.Constraint(expr=model.e_bess[n - 1] >= soc_start * model.E_N)

    model.grid_balance = opt.Constraint(
        model.T, rule=lambda m, t: m.p_grid[t] == m.p_load[t] + m.p_ch[t] - m.p_dis[t]
    )
    model.peak_definition = opt.Constraint(model.T, rule=lambda m, t: m.p_grid[t] <= m.p_peak)

    energiebezug_kwh = sum(model.p_grid[t] * dt_h for t in model.T)
    stromkosten_expr = (
        stromkosten["arbeitspreis_eur_kwh"] * energiebezug_kwh
        + stromkosten["leistungspreis_eur_kw_a"] * model.p_peak
    )
    speicherjahreskosten_expr = af * (
        speicher_params["fixkosten_eur"] * model.storage_installed
        + speicher_params["kapazitaetskosten_eur_kwh"] * model.E_N
        + speicher_params["leistungskosten_eur_kw"] * model.P_N
    )

    model.obj = opt.Objective(expr=stromkosten_expr + speicherjahreskosten_expr, sense=opt.minimize)
    return model


In [ ]:
model_kapitel_2 = build_model_optimale_peakgrenze(
    load_profile=profile,
    stromkosten=stromkosten,
    speicher_params=speicher_standard,
    grenzen=dimensionierungsgrenzen,
    dt_h=dt,
)
loese_modell(model_kapitel_2)

res_kapitel_2 = ergebnisse_aus_modell(model_kapitel_2, profile)

kapitel_2_kapazitaet = opt.value(model_kapitel_2.E_N)
kapitel_2_leistung = opt.value(model_kapitel_2.P_N)
kapitel_2_peak = opt.value(model_kapitel_2.p_peak)
kapitel_2_kosten_mit_speicher = stromrechnung(res_kapitel_2["Netzbezug_kW"], stromkosten, dt)

print(f"Optimale Peak-Shaving-Grenze: {kapitel_2_peak:,.2f} kW")
wirtschaftlichkeit_kapitel_2 = drucke_wirtschaftlichkeit(
    name="Kapitel 2: optimale Peak-Shaving-Grenze",
    ohne_speicher=kosten_ohne_speicher,
    mit_speicher=kapitel_2_kosten_mit_speicher,
    kapazitaet_kwh=kapitel_2_kapazitaet,
    leistung_kw=kapitel_2_leistung,
    params=speicher_standard,
)


In [ ]:
plot_peak_shaving_ergebnis(res_kapitel_2, "Kapitel 2")


## Kapitel 3 – Vergleich zweier Speichertechnologien

Im letzten Kapitel wird die Modelllogik aus Kapitel 2 unverändert genutzt, aber mit zwei unterschiedlichen Speichertechnologien ausgeführt. Dadurch wird sichtbar, wie stark die optimale Peak-Shaving-Grenze von Technologieparametern abhängt.

Die gewählten Parameter sind bewusst idealtypisch:

- **Lithium-Ionen-Batterie:** höhere spezifische Kapazitätskosten, aber gute Wirkungsgrade und moderate Leistungskosten.
- **Redox-Flow-Batterie:** niedrigere spezifische Kapazitätskosten, aber höhere Leistungskosten und geringere Wirkungsgrade.

Für beide Technologien berechnet die Optimierung:

- optimale Speicherkapazität,
- optimale Speicherleistung,
- optimale Peak-Shaving-Grenze,
- jährliche Stromkosten,
- annualisierte Speicherkosten,
- statische Amortisationsdauer.


In [ ]:
technologien = {
    "Lithium-Ionen-Batterie": {
        "fixkosten_eur": 50_000.0,
        "kapazitaetskosten_eur_kwh": 300.0,
        "leistungskosten_eur_kw": 150.0,
        "soc_bounds": (0.05, 0.95),
        "soc_start": 0.10,
        "eta_laden": 0.95,
        "eta_entladen": 0.95,
        "nutzungsdauer_jahre": 20,
        "kalkulationszins": 0.05,
    },
    "Redox-Flow-Batterie": {
        "fixkosten_eur": 30_000.0,
        "kapazitaetskosten_eur_kwh": 150.0,
        "leistungskosten_eur_kw": 300.0,
        "soc_bounds": (0.05, 1.00),
        "soc_start": 0.10,
        "eta_laden": 0.82,
        "eta_entladen": 0.82,
        "nutzungsdauer_jahre": 30,
        "kalkulationszins": 0.05,
    },
}

parameter_tabelle = pd.DataFrame(technologien).T[
    [
        "fixkosten_eur",
        "kapazitaetskosten_eur_kwh",
        "leistungskosten_eur_kw",
        "eta_laden",
        "eta_entladen",
        "nutzungsdauer_jahre",
        "kalkulationszins",
    ]
]
parameter_tabelle


In [ ]:
vergleich_zeilen = []
vergleich_ergebnisse = {}

for name, params in technologien.items():
    print(f"\nOptimiere Technologie: {name}")
    model = build_model_optimale_peakgrenze(
        load_profile=profile,
        stromkosten=stromkosten,
        speicher_params=params,
        grenzen=dimensionierungsgrenzen,
        dt_h=dt,
    )
    loese_modell(model)
    df = ergebnisse_aus_modell(model, profile)

    kapazitaet = opt.value(model.E_N)
    leistung = opt.value(model.P_N)
    peak = opt.value(model.p_peak)
    kosten_mit = stromrechnung(df["Netzbezug_kW"], stromkosten, dt)
    capex = investitionskosten_speicher(kapazitaet, leistung, params)
    jahreskosten = jahreskosten_speicher(kapazitaet, leistung, params)
    stromkostenersparnis = kosten_ohne_speicher["stromkosten_eur_a"] - kosten_mit["stromkosten_eur_a"]
    netto_vorteil = stromkostenersparnis - jahreskosten
    amortisation = amortisationsdauer_jahre(capex, stromkostenersparnis)

    vergleich_ergebnisse[name] = {"model": model, "df": df, "params": params}
    vergleich_zeilen.append(
        {
            "Technologie": name,
            "Optimale Peak-Grenze [kW]": peak,
            "Kapazität [kWh]": kapazitaet,
            "Leistung [kW]": leistung,
            "Investition [€]": capex,
            "Speicherjahreskosten [€/a]": jahreskosten,
            "Stromkosten mit Speicher [€/a]": kosten_mit["stromkosten_eur_a"],
            "Stromkostenersparnis [€/a]": stromkostenersparnis,
            "Netto-Vorteil [€/a]": netto_vorteil,
            "Amortisationsdauer [a]": amortisation,
        }
    )

vergleich = pd.DataFrame(vergleich_zeilen).set_index("Technologie")
vergleich.round(2)


In [ ]:
# Gemeinsamer Vergleich der Netzbezugsleistung
vergleich_plot = pd.DataFrame(index=profile.index)
vergleich_plot["Residuallast ohne Speicher [kW]"] = profile["pRes"]
for name, result in vergleich_ergebnisse.items():
    vergleich_plot[f"Netzbezug {name} [kW]"] = result["df"]["Netzbezug_kW"]

vergleich_plot.plot(
    template=template,
    labels={"value": "Leistung [kW]", "time_date": "Zeit", "variable": "Zeitreihe"},
    title="Technologievergleich: Netzbezug und optimale Peak-Shaving-Grenzen",
)


In [ ]:
# Vergleich der Speicherleistungen
leistung_plot = pd.DataFrame(index=profile.index)

plot_reihenfolge = [
    "Redox-Flow-Batterie",
    "Lithium-Ionen-Batterie",
]

for name in plot_reihenfolge:
    result = vergleich_ergebnisse[name]
    leistung_plot[f"BESS-Leistung {name} [kW]"] = result["df"]["BESS_Leistung_kW"]

fig = leistung_plot.plot(
    template=template,
    labels={
        "value": "Leistung [kW]",
        "time_date": "Zeit",
        "variable": "Zeitreihe",
    },
    title="Technologievergleich: Speicherleistung, positiv = Laden, negativ = Entladen",
)

fig.show()

### Einordnung des Technologievergleichs

Der Vergleich zeigt, dass Peak Shaving nicht nur eine Frage des Betriebs, sondern auch eine Frage der Speichertechnologie ist. Eine Technologie mit niedrigen Kapazitätskosten kann größere Energiemengen wirtschaftlicher bereitstellen, während niedrige Leistungskosten vor allem bei kurzen, hohen Spitzen vorteilhaft sind.

Die optimale Peak-Shaving-Grenze ergibt sich daher nicht aus einer technischen Faustregel, sondern aus dem Zusammenspiel von:

- Form und Häufigkeit der Lastspitzen,
- Arbeitspreis und Leistungspreis,
- Wirkungsgradverlusten des Speichers,
- Fixkosten,
- spezifischen Kapazitätskosten,
- spezifischen Leistungskosten,
- angenommener Nutzungsdauer und Kapitalzins.

Da in diesem Notebook ein Jahr als repräsentatives Beispieljahr verwendet wird, werden die jährlichen Stromkosteneffekte aus genau diesem Jahr berechnet. Für eine mehrjährige Betrachtung kann dasselbe Profil vereinfacht für jedes Jahr wiederholt werden. Die statische Amortisationsdauer vergleicht die einmalige Investition mit der jährlichen Stromkostenersparnis.
